# 02 Web后端开发 · 第02课：FastAPI入门

上一课使用函数和字典模拟了HTTP请求、路由和响应。这一课开始使用FastAPI创建一个真正能被浏览器或其他程序访问的后端服务器。

## 本课目标

1. 理解Web框架替我们完成了什么；
2. 创建并启动FastAPI应用；
3. 理解路由装饰器；
4. 编写GET和POST接口；
5. 区分路径参数、查询参数和请求体；
6. 使用Swagger页面手动测试接口；
7. 使用`TestClient`编写自动化接口测试。

## 1. 为什么需要FastAPI？

上一课的`route_request()`只是普通Python函数，并没有真的监听网络端口。真正的后端还需要处理网络连接、解析HTTP请求、匹配路由、转换JSON和构造HTTP响应。

FastAPI是Python Web框架，它替我们处理这些通用工作。我们主要编写每个接口真正要完成的业务逻辑。

```text
客户端发送HTTP请求
        ↓
Uvicorn接收网络连接
        ↓
FastAPI解析请求并匹配路由
        ↓
执行我们编写的Python函数
        ↓
FastAPI把返回值转换成JSON响应
```

- **FastAPI**：Web框架，负责路由、参数解析、数据验证和响应生成；
- **Uvicorn**：ASGI服务器，负责监听端口并接收真正的网络请求。

二者分工不同，但通常一起使用。

## 2. 安装本课依赖

在本课文件夹打开PowerShell，运行：

```powershell
python -m pip install -r requirements.txt
```

命令说明：

- `python -m pip`：使用当前Python解释器运行pip，避免把包装到另一个Python环境；
- `install`：pip的安装命令；
- `-r requirements.txt`：`-r`表示从指定文件读取需要安装的依赖。

本课使用：

- `fastapi`：创建Web应用；
- `uvicorn`：启动开发服务器；
- `httpx`：`TestClient`底层使用的HTTP客户端；
- `pytest`：运行自动化测试。

## 3. 创建第一个FastAPI应用

下面的代码适合放在单独的`.py`文件中。Notebook用于逐段阅读和理解；真正启动服务器时请运行同目录下的Python文件。

In [1]:
# FastAPI类来自第三方库fastapi，需要先安装
from fastapi import FastAPI

# FastAPI(title="入门示例")：创建一个FastAPI应用实例
#   title参数：接口文档中显示的应用名称
#   返回值：FastAPI对象，保存到变量app中
# app这个变量名是约定俗成的，也可以使用其他合法变量名
app = FastAPI(title="入门示例")

# app.get("/ping")：注册一个GET路由
#   path参数：客户端访问的URL路径，这里是/ping
#   返回值：一个装饰器；它会把紧随其后的函数登记为该路由的处理函数
@app.get("/ping")
def ping() -> dict[str, str]:
    # FastAPI会自动把Python字典转换成JSON响应
    return {"message": "pong"}

## 4. `@app.get()`为什么写在函数上面？

`@`开头的语法叫装饰器。这里可以先把它理解为“给下面的函数登记一个访问地址”。

```python
@app.get("/ping")
def ping():
    return {"message": "pong"}
```

它表达的是：收到`GET /ping`请求时，请执行`ping()`。

装饰器下面必须紧跟要注册的函数。装饰器只负责登记关系，客户端请求到来时FastAPI才会调用该函数。

路由通常由两部分共同确定：

```text
HTTP方法 + 路径
GET        /ping
POST       /ping
```

路径相同但方法不同，仍然是不同接口。

## 5. 启动服务器

假设上面的代码保存在`demo.py`，进入该文件所在目录后运行：

```powershell
python -m uvicorn demo:app --reload
```

每部分的含义：

- `python -m uvicorn`：使用当前Python环境运行Uvicorn；
- `demo:app`：冒号左侧`demo`表示`demo.py`模块，右侧`app`表示文件中的FastAPI对象；
- `--reload`：代码文件变化后自动重启服务器，适合本地学习，不用于正式生产环境；
- 默认主机是`127.0.0.1`，表示只有本机可以访问；
- 默认端口是`8000`。

启动后可访问：

- `http://127.0.0.1:8000/ping`：调用接口；
- `http://127.0.0.1:8000/docs`：Swagger交互式接口文档；
- `http://127.0.0.1:8000/redoc`：另一种自动生成的接口文档。

终端需要保持运行，服务器才能继续接受请求。按`Ctrl+C`可以停止服务器。

## 6. 查询参数

没有写进路径模板、但出现在处理函数参数中的简单类型参数，通常会被FastAPI当作查询参数。

In [ ]:
@app.get("/welcome")
def welcome(name: str = "朋友") -> dict[str, str]:
    return {"message": f"你好，{name}"}

# 访问/welcome时，name使用默认值"朋友"
# 访问/welcome?name=小明时，name收到字符串"小明"

`name: str = "朋友"`同时表达了三件事：

- 参数名是`name`；
- FastAPI应把它按字符串处理；
- 客户端没有提供时使用默认值`"朋友"`。

这里没有额外调用库函数。FastAPI会读取函数签名，自动完成参数提取和基本类型验证。

## 7. 路径参数

路径模板中的`{变量名}`表示路径参数，对应名称必须出现在处理函数参数中。

In [ ]:
@app.get("/products/{product_id}")
def get_product(product_id: int) -> dict[str, int]:
    return {"product_id": product_id}

# 访问/products/25时，FastAPI会把路径中的"25"转换成整数25
# 如果访问/products/abc，因为abc不能转换成整数，FastAPI会自动返回422

路径参数和查询参数的判断关键：

```text
@app.get("/products/{product_id}")
                         ↑ 出现在路径模板中 → 路径参数

def get_product(product_id: int, detail: bool = False):
                                ↑ 没出现在路径中 → 查询参数
```

对应URL可以是：`/products/25?detail=true`。

## 8. POST请求与请求体

GET通常用于查询，POST通常用于创建资源或提交需要处理的数据。FastAPI可以把JSON请求体解析成Python字典。

In [ ]:
from typing import Any

# Any来自Python标准库typing，用于类型标注，表示值可以是任意类型

# app.post("/mirror", status_code=201)：注册POST路由
#   path参数：接口路径/mirror
#   status_code参数：成功响应默认使用的HTTP状态码，这里是201
#   返回值：登记下方处理函数的装饰器
@app.post("/mirror", status_code=201)
def mirror(payload: dict[str, Any]) -> dict[str, Any]:
    # payload没有出现在路径中，并且是复杂数据类型，因此FastAPI把它当作JSON请求体
    return {"received": payload}

在Swagger页面中调用`POST /mirror`时，可以提交：

```json
{
  "text": "你好",
  "count": 2
}
```

FastAPI把JSON对象转换成Python字典并赋给`payload`。函数返回的Python字典又会被转换成JSON响应。

这一课使用普通字典只是为了观察请求体。下一课会学习Pydantic，用明确的数据模型规定请求体必须有哪些字段。

## 9. 使用`TestClient`测试接口

自动化测试不需要真的在终端启动Uvicorn。FastAPI提供的`TestClient`可以在测试代码中模拟客户端请求。

In [2]:
# TestClient类来自fastapi.testclient
from fastapi.testclient import TestClient

# TestClient(app)：创建测试客户端
#   app参数：要测试的FastAPI应用对象
#   返回值：TestClient对象
client = TestClient(app)

# client.get(url, params=None)：模拟GET请求
#   url参数：请求路径
#   params参数：可选的查询参数字典
#   返回值：Response响应对象
response = client.get("/welcome", params={"name": "小明"})

# response.status_code属性：HTTP状态码
# response.json()：把JSON响应体解析成Python对象；此方法不需要参数
print(response.status_code)
print(response.json())

404
{'detail': 'Not Found'}


<Python环境>/Lib/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


POST请求测试示例：

```python
# client.post(url, json=None)：模拟POST请求
#   url参数：请求路径
#   json参数：作为JSON请求体发送的Python对象
#   返回值：Response响应对象
response = client.post("/mirror", json={"text": "你好"})
```

`json=`和响应的`.json()`不是一回事：前者表示发送JSON请求体，后者表示读取JSON响应体。

## 10. 关键关系补充：把整个运行过程串起来

### 10.1 FastAPI和Uvicorn不是同一个东西

`app = FastAPI()`创建的是一个Web应用对象。它保存路由规则，并知道怎样解析参数、验证数据、执行处理函数和生成响应，但它本身不会主动占用`8000`端口等待浏览器连接。

Uvicorn是一个ASGI服务器程序。这里可以先把ASGI理解为服务器与Python Web应用之间的标准沟通方式。Uvicorn负责：

1. 启动一个持续运行的进程；
2. 监听指定主机和端口；
3. 接收真实HTTP请求；
4. 按ASGI规则把请求交给FastAPI应用；
5. 把FastAPI生成的响应发回客户端。

因此两者的关系是：

```text
浏览器或其他客户端
        ↓ 真实网络请求
Uvicorn（接收连接、监听端口）
        ↓ ASGI调用
FastAPI应用（路由、验证、业务处理、响应）
```

### 10.2 `app:app`到底表示什么？

命令：

```powershell
python -m uvicorn app:app --reload
```

其中第一个`app`表示Python模块`app.py`，第二个`app`表示该模块中的变量：

```python
# app.py
app = FastAPI()
```

Uvicorn会做近似于`from app import app`的事情，找到这个FastAPI对象并运行它。如果文件叫`demo.py`，变量仍叫`app`，就应写`demo:app`。

### 10.3 装饰器在什么时候工作，处理函数在什么时候执行？

Python导入模块时，`@app.get("/health")`会把“GET方法、/health路径、health函数”登记到FastAPI的路由表中。这个阶段只是登记，不代表已经有人访问接口。

客户端以后真正发送`GET /health`时，FastAPI查找路由表，匹配成功后才调用`health()`。

```text
程序启动：登记 GET /health → health
收到请求：匹配 GET /health → 执行 health()
```

### 10.4 FastAPI如何判断参数来自哪里？

本课先使用下面的基础判断规则：

| 函数参数情况 | FastAPI通常把它当作 |
|---|---|
| 名称出现在路径模板`{}`中 | 路径参数 |
| 简单类型参数未出现在路径模板中 | 查询参数 |
| `dict`等复杂数据参数用于POST接口 | JSON请求体 |

例如`/items/{item_id}`明确把`item_id`写进了URL路径模板，所以它是路径参数。`/hello?name=小明`中的`name=小明`位于`?`后面，因此它是查询参数。参数是不是查询参数，与后端有没有查询数据库没有关系。

### 10.5 `422`发生在处理函数执行之前

对于`item_id: int`，FastAPI会先尝试把URL中的文本转换成整数。`/items/25`可以成功转换；`/items/abc`无法转换。验证失败时，FastAPI直接返回`422`，不会调用`get_item()`。

### 10.6 `json=`和`.json()`方向相反

```python
response = client.post("/echo", json=data)
```

`client.post(url, json=data)`来自测试客户端：

- `url`参数指定请求路径；
- `json`参数接收要发送的Python对象；
- 测试客户端把该对象编码成JSON请求体，并设置JSON内容类型；
- 返回一个`Response`响应对象。

```python
result = response.json()
```

`response.json()`是响应对象的方法，不接收参数。它读取服务器返回的JSON响应体，并解析成Python字典、列表等对象。

```text
json=data        Python对象 → JSON请求体 → 发给应用
response.json()  JSON响应体 → Python对象 → 交给测试代码
```

### 10.7 为什么TestClient不需要Uvicorn？

`TestClient(app)`已经直接拿到了FastAPI应用对象。它在当前Python进程内部模拟请求，并按照ASGI方式直接调用应用，不需要打开真实网络端口。

它仍会经过FastAPI的路由、参数验证和响应生成，所以适合自动化测试；但如果要让浏览器或其他电脑通过真实网络访问，仍然需要Uvicorn。

```text
浏览器测试：浏览器 → 网络 → Uvicorn → FastAPI
TestClient：测试代码 → TestClient → FastAPI
```


## 11. 本课作业

请完成同目录`app.py`中的四个接口。题目只规定接口契约，具体Python实现由你决定。

| 方法与路径 | 输入 | 成功状态码 | JSON响应体 |
|---|---|---|---|
| `GET /health` | 无 | 200 | `{"status": "ok"}` |
| `GET /hello` | 可选查询参数`name`，默认值为`世界` | 200 | `{"message": "你好，实际名称"}` |
| `GET /items/{item_id}` | 整数路径参数`item_id` | 200 | `{"item_id": 实际整数}` |
| `POST /echo` | 任意JSON对象请求体 | 201 | `{"received": 原请求体}` |

说明：

- “实际名称”指客户端传入的`name`；未传时使用`世界`；
- “原请求体”表示响应中保留客户端提交的数据，不要写死具体内容；
- 路径参数无法转换成整数时，应由FastAPI自动返回`422`；
- 不要修改函数名、参数、装饰器路径和测试文件。

运行测试：

```powershell
pytest -v
```

最终目标是`8 passed`。

## 12. 理解题（请把答案写在每题下面）

1. FastAPI和Uvicorn分别负责什么？
- FastAPI负责路由、参数验证、执行处理函数和生成响应；
- Uvicorn负责监听网络端口、接收真实HTTP请求，并把请求交给FastAPI。
2. `app = FastAPI()`创建的`app`是什么？为什么启动命令里要写`app:app`？
- `app`是一个FastAPI应用对象，保存应用配置、路由和参数验证等信息。
- `app:app`中第一个`app`表示`app.py`模块，第二个`app`表示该模块中的FastAPI应用变量。

3. `@app.get("/health")`的作用是什么？它会在程序启动时直接执行下面的函数吗？
- 告诉服务器执行的是get方法，且路由为/health
- 不会，只有在客户端访问/health的时候才会执行

4. `/items/{item_id}`中的`item_id`为什么属于路径参数？
- 因为`item_id`以`{item_id}`的形式直接出现在路由路径模板中，FastAPI会从URL路径对应位置提取它，所以它属于路径参数。

5. `/hello?name=小明`中的`name`为什么属于查询参数？
- 因为这个参数位于URL的?后

6. `client.post(..., json=data)`中的`json=`与`response.json()`分别做什么？
- 把data中的数据转化成JSON格式发给FastAPI
- 读取服务器返回的JSON响应体，并解析成Python对象。

7. 为什么访问`/items/abc`可能返回`422`，而不是让处理函数正常得到一个字符串？
- 因为FastAPI尝试将abc转换成int类型，但是失败了

8. 为什么自动化测试通常不需要先手动运行Uvicorn？
- 因为TestClient直接持有FastAPI应用对象，可以在Python进程内部模拟请求并调用应用，不需要打开网络端口，所以不需要Uvicorn。